# 02 · DistilBERT fine-tuning

**Stage 3** of the plan. We fine-tune `distilbert-base-uncased` on the **same splits** as the TF‑IDF baseline
(made in notebook 01), with **3 seeds**, then compare the two models on the clean test set.

**Before running:** Runtime → *Change runtime type* → **T4 GPU**. Then Runtime → *Run all*.
Expect roughly 5–10 minutes per seed on a T4.

**If Colab disconnects:** just run all again. Seeds that already finished are saved to Drive and skipped.

### ▶ Keep this tab awake (tap play once)

Phone browsers suspend background tabs, which can freeze the notebook's progress display or drop the
connection while long cells run. The player below loops a **near-silent** 5‑second sound. Tap ▶ once
after starting *Run all*; it keeps playing in your browser while the other cells run.
It does **not** extend Colab's own runtime limits, so keep the phone charging and Colab open.

In [ ]:
import io, wave, base64
import numpy as np
from IPython.display import HTML, display

_sr = 8000
_noise = (np.random.default_rng(0).standard_normal(_sr * 5) * 20).astype(np.int16)   # ~-64 dBFS: inaudible
_buf = io.BytesIO()
with wave.open(_buf, "wb") as _w:
    _w.setnchannels(1); _w.setsampwidth(2); _w.setframerate(_sr); _w.writeframes(_noise.tobytes())
display(HTML(f'<audio controls loop autoplay src="data:audio/wav;base64,{base64.b64encode(_buf.getvalue()).decode()}"></audio>'
             '<div style="font-size:12px;color:gray">Keep-alive player: tap ▶ if it is not already playing.</div>'))

## 0 · Configuration

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256               # tokens for title + text; longer articles are truncated
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 128
EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
SEEDS = [42, 43, 44]
PROJECT_DIRNAME = "NLP_style_attack"

## 1 · Setup

In [ ]:
import os, sys, json, time, random, copy
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = f"/content/drive/MyDrive/{PROJECT_DIRNAME}"
else:
    BASE_DIR = os.environ.get("FAKESTYLE_BASE_DIR", os.path.abspath(f"./{PROJECT_DIRNAME}"))
DIRS = {k: os.path.join(BASE_DIR, k) for k in ["data", "results", "models", "figures"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU found. Switch to a T4 runtime (Runtime -> Change runtime type); "
          "training on CPU would take many hours.")
print("Outputs ->", BASE_DIR)

## 2 · Load the splits from notebook 01

In [ ]:
splits = {}
for name in ["train", "val", "test"]:
    path = os.path.join(DIRS["data"], f"{name}.parquet")
    assert os.path.exists(path), f"{path} not found - run notebook 01 first"
    splits[name] = pd.read_parquet(path)
print(pd.DataFrame({n: {"articles": len(d), "% fake": round(100 * d["fake"].mean(), 1)} for n, d in splits.items()}).T)

## 3 · Tokenise

Input is the pair `(title, text)`, which the tokenizer joins as `[CLS] title [SEP] text [SEP]`.
When the pair is longer than `MAX_LEN`, the longer part (almost always the text) is cut first.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(d):
    return tokenizer(d["title"].tolist(), d["text"].tolist(), truncation="longest_first", max_length=MAX_LEN)

t0 = time.time()
encoded = {n: encode(d) for n, d in splits.items()}
print(f"Tokenised in {time.time() - t0:.0f}s")

full_len = np.array([len(x) for x in tokenizer(splits["test"]["title"].tolist(), splits["test"]["text"].tolist(),
                                                truncation=False)["input_ids"]])
print(f"Test set: median {np.median(full_len):.0f} tokens; {100 * (full_len > MAX_LEN).mean():.1f}% of articles "
      f"are truncated to {MAX_LEN} tokens")

## 4 · Training loop

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

class EncodedDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc, self.labels = enc, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in ("input_ids", "attention_mask")}
        item["labels"] = int(self.labels[i])
        return item

def collate(batch):
    return tokenizer.pad(batch, return_tensors="pt")

datasets = {n: EncodedDataset(encoded[n], splits[n]["fake"].values) for n in splits}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def evaluate(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "precision_fake": precision_score(y_true, y_pred, pos_label=1),
        "recall_fake": recall_score(y_true, y_pred, pos_label=1),
        "precision_real": precision_score(y_true, y_pred, pos_label=0),
        "recall_real": recall_score(y_true, y_pred, pos_label=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "n": int(len(y_true)),
    }

@torch.no_grad()
def predict_proba(model, dataset):
    model.eval()
    probs = []
    for batch in DataLoader(dataset, batch_size=EVAL_BATCH_SIZE, collate_fn=collate):
        batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
        probs.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(probs)

def train_one_seed(seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                        generator=torch.Generator().manual_seed(seed))
    no_decay = ("bias", "LayerNorm.weight")
    params = [
        {"params": [p for n, p in model.named_parameters() if not n.endswith(no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(params, lr=LR)
    total_steps = EPOCHS * len(loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRAC * total_steps), total_steps)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

    best_f1, best_state, history = -1.0, None, []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t0, running = time.time(), 0.0
        for step, batch in enumerate(loader, 1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = model(**batch).loss
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            running += loss.item()
            if step % 100 == 0 or step == len(loader):
                print(f"  seed {seed} · epoch {epoch} · step {step}/{len(loader)} · loss {running / step:.4f} "
                      f"· {time.time() - t0:.0f}s", flush=True)
        val_prob = predict_proba(model, datasets["val"])
        val_f1 = f1_score(splits["val"]["fake"].values, (val_prob >= 0.5).astype(int), average="macro")
        history.append({"epoch": epoch, "train_loss": running / len(loader), "val_macro_f1": val_f1})
        print(f"  -> epoch {epoch}: val macro-F1 = {val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, history

## 5 · Train 3 seeds

For each seed we keep the epoch with the best validation macro‑F1, then save to Drive:
- the model (`models/distilbert_seed{N}/`), reused by the style-attack notebook
- per-article probabilities for val and test (`results/02_preds_seed{N}.parquet`), used for paired significance tests
- metrics and training history (`results/02_seed{N}.json`)

A seed that already has all three is skipped.

In [ ]:
seed_results = {}
for seed in SEEDS:
    model_dir = os.path.join(DIRS["models"], f"distilbert_seed{seed}")
    preds_path = os.path.join(DIRS["results"], f"02_preds_seed{seed}.parquet")
    json_path = os.path.join(DIRS["results"], f"02_seed{seed}.json")
    if all(os.path.exists(p) for p in (model_dir, preds_path, json_path)):
        print(f"seed {seed}: already done, loading saved results")
        with open(json_path) as f:
            seed_results[seed] = json.load(f)
        continue

    print(f"=== seed {seed} ===")
    t_start = time.time()
    model, history = train_one_seed(seed)
    out, pred_frames = {"seed": seed, "history": history}, []
    for name in ["val", "test"]:
        prob = predict_proba(model, datasets[name])
        y = splits[name]["fake"].values
        out[name] = {k: float(v) for k, v in evaluate(y, (prob >= 0.5).astype(int), prob).items()}
        pred_frames.append(pd.DataFrame({"id": splits[name]["id"].values, "split": name, "fake": y, "prob_fake": prob}))
    out["train_minutes"] = round((time.time() - t_start) / 60, 1)

    model.save_pretrained(model_dir); tokenizer.save_pretrained(model_dir)
    pd.concat(pred_frames).to_parquet(preds_path)
    with open(json_path, "w") as f:
        json.dump(out, f, indent=2)
    seed_results[seed] = out
    print(f"seed {seed}: test accuracy {out['test']['accuracy']:.4f}, macro-F1 {out['test']['macro_f1']:.4f} "
          f"({out['train_minutes']} min)\n")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 6 · Results across seeds

In [ ]:
rows = {seed: r["test"] for seed, r in seed_results.items()}
per_seed = pd.DataFrame(rows).T.drop(columns="n")
agg = pd.DataFrame({"mean": per_seed.mean(), "std": per_seed.std(ddof=1)}).T
print("DistilBERT · test set, per seed:\n", per_seed.round(4), "\n")
print("Mean ± std over seeds:\n", agg.round(4))

## 7 · Comparison with the TF‑IDF baseline

Both models are scored on the **same 4,500 test articles**. McNemar's test (exact, on the paired correct/incorrect
outcomes) checks whether the difference in errors is significant. DistilBERT seed 42 is used for the paired test.

In [ ]:
import joblib
from scipy.stats import binomtest

with open(os.path.join(DIRS["results"], "01_summary.json")) as f:
    tfidf_summary = json.load(f)
tfidf = joblib.load(os.path.join(DIRS["models"], "tfidf_lr.joblib"))
test = splits["test"]
tfidf_prob = tfidf["clf"].predict_proba(tfidf["vectorizer"].transform((test["title"] + "\n\n" + test["text"]).tolist()))[:, 1]
y_test = test["fake"].values

ref_seed = SEEDS[0]
bert_preds = pd.read_parquet(os.path.join(DIRS["results"], f"02_preds_seed{ref_seed}.parquet"))
bert_prob = bert_preds[bert_preds["split"] == "test"].set_index("id").loc[test["id"], "prob_fake"].values

tfidf_correct = (tfidf_prob >= 0.5).astype(int) == y_test
bert_correct = (bert_prob >= 0.5).astype(int) == y_test
b = int((tfidf_correct & ~bert_correct).sum())    # TF-IDF right, DistilBERT wrong
c = int((~tfidf_correct & bert_correct).sum())    # DistilBERT right, TF-IDF wrong
mcnemar_p = binomtest(b, b + c, 0.5).pvalue if b + c > 0 else 1.0

comparison = pd.DataFrame({
    "TF-IDF + LR": tfidf_summary["metrics"]["test"],
    "DistilBERT (mean of seeds)": agg.loc["mean"].to_dict(),
    "DistilBERT (std)": agg.loc["std"].to_dict(),
}).T.drop(columns="n", errors="ignore")
print(comparison.round(4), "\n")
print(f"Paired on the test set: TF-IDF right & DistilBERT wrong = {b}, DistilBERT right & TF-IDF wrong = {c}")
print(f"McNemar exact p-value = {mcnemar_p:.3g}")

### 7.1 · Where the models disagree

A few test articles that one model gets right and the other gets wrong. Useful starting points for the
error analysis in Stage 4.

In [ ]:
pd.set_option("display.max_colwidth", 140)
show = test.assign(tfidf_prob=tfidf_prob.round(3), bert_prob=bert_prob.round(3))
for title_, mask in [("TF-IDF right, DistilBERT wrong", tfidf_correct & ~bert_correct),
                     ("DistilBERT right, TF-IDF wrong", ~tfidf_correct & bert_correct)]:
    print(f"===== {title_} ({mask.sum()}) =====")
    for _, r in show[mask].head(4).iterrows():
        print(f"[{r['id']}] {'FAKE' if r['fake'] else 'REAL'} · tfidf p(fake)={r['tfidf_prob']} · bert p(fake)={r['bert_prob']}")
        print(f"    {r['title'][:110]}\n    {r['text'][:220]}...\n")

## 8 · Save summary

In [ ]:
import transformers, sklearn
summary = {
    "notebook": "02_distilbert",
    "config": {"MODEL_NAME": MODEL_NAME, "MAX_LEN": MAX_LEN, "BATCH_SIZE": BATCH_SIZE, "EPOCHS": EPOCHS, "LR": LR,
               "WEIGHT_DECAY": WEIGHT_DECAY, "WARMUP_FRAC": WARMUP_FRAC, "SEEDS": SEEDS,
               "torch": torch.__version__, "transformers": transformers.__version__,
               "device": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"},
    "test_truncated_pct": round(float(100 * (full_len > MAX_LEN).mean()), 1),
    "per_seed_test": {str(s): {k: round(v, 4) for k, v in r["test"].items()} for s, r in seed_results.items()},
    "per_seed_history": {str(s): r["history"] for s, r in seed_results.items()},
    "test_mean": {k: round(float(v), 4) for k, v in agg.loc["mean"].items()},
    "test_std": {k: round(float(v), 4) for k, v in agg.loc["std"].items()},
    "vs_tfidf": {"tfidf_test": tfidf_summary["metrics"]["test"], "mcnemar_seed": ref_seed,
                 "tfidf_right_bert_wrong": b, "bert_right_tfidf_wrong": c, "mcnemar_p": float(mcnemar_p)},
}
with open(os.path.join(DIRS["results"], "02_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

**Done.** New in `MyDrive/NLP_style_attack/`:

| Path | Contents |
|---|---|
| `models/distilbert_seed{42,43,44}/` | Fine-tuned models + tokenizer, reused for the style attack |
| `results/02_preds_seed*.parquet` | Per-article P(fake) on val and test |
| `results/02_seed*.json`, `results/02_summary.json` | Metrics, training history, comparison with TF‑IDF |

Next: **03 · Style attack**.